# 주인을 알아보는 로봇청소기 — 얼굴인식 AI 12단계 실습
Google Colab에서 위에서 아래로 순서대로 실행합니다. **본인 동의를 받은 사진만 사용**하고 얼굴사진과 임베딩은 GitHub에 올리지 마세요.
사전학습된 FaceNet의 특징벡터를 평균내는 실습이며, FaceNet 가중치를 새로 학습하지 않습니다.


## 01. 01 · Colab 환경 확인


In [ ]:
import sys
print("Python:", sys.version)
import torch
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

## 02. 02 · 필수 라이브러리 설치


In [ ]:
%pip install -q facenet-pytorch opencv-python-headless Pillow
# Colab 런타임에서 torch/torchvision 충돌 시 런타임 재시작 후 다시 실행하세요.

## 03. 03 · 라이브러리 불러오기


In [ ]:
from pathlib import Path
import zipfile
import torch
import torch.nn.functional as F
from PIL import Image
from facenet_pytorch import MTCNN, InceptionResnetV1
from google.colab import files
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 04. 04 · faces.zip 업로드


In [ ]:
uploaded = files.upload()
assert "faces.zip" in uploaded, "faces.zip 파일을 선택하세요."
zip_path = Path("faces.zip")

## 05. 05 · owner 얼굴사진 8장 확인


In [ ]:
target = Path("data")
target.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_path) as archive:
    for item in archive.infolist():
        path = Path(item.filename)
        if item.is_dir():
            continue
        if path.is_absolute() or ".." in path.parts:
            raise ValueError("Unsafe ZIP path")
        if path.parts[0] not in {"owner", "faces"}:
            raise ValueError("ZIP에는 owner/ 폴더를 넣으세요.")
    archive.extractall(target)
owner_dir = target / "owner"
if not owner_dir.exists():
    owner_dir = target / "faces" / "owner"
photos = sorted(p for p in owner_dir.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
print("얼굴사진 수:", len(photos), [p.name for p in photos])
assert len(photos) >= 1, "owner 폴더의 사진을 확인하세요."

## 06. 06 · MTCNN과 FaceNet 준비


In [ ]:
detector = MTCNN(image_size=160, margin=0, keep_all=False, device=device)
facenet = InceptionResnetV1(pretrained="vggface2").eval().to(device)
print("얼굴 검출기와 사전 학습 FaceNet 로딩 완료")

## 07. 07 · 사진에서 얼굴 검출


In [ ]:
detected = []
for path in photos:
    with Image.open(path) as image:
        face = detector(image.convert("RGB"))
    if face is not None:
        detected.append((path.name, face))
        print("검출 성공:", path.name)
    else:
        print("검출 실패:", path.name)
assert detected, "얼굴 검출에 성공한 사진이 없습니다."

## 08. 08 · 512차원 얼굴 특징벡터 추출


In [ ]:
vectors = []
with torch.no_grad():
    for name, face in detected:
        embedding = facenet(face.unsqueeze(0).to(device)).squeeze(0).cpu()
        embedding = F.normalize(embedding, dim=0)
        vectors.append(embedding)
print("특징벡터 수:", len(vectors), "차원:", vectors[0].numel())

## 09. 09 · 주인 대표벡터 만들기


In [ ]:
# 사전 학습된 FaceNet은 고정하고, 8장 특징벡터의 평균으로 주인 프로필을 구성합니다.
owner_embedding = F.normalize(torch.stack(vectors).mean(dim=0), dim=0)
print("대표벡터 크기:", tuple(owner_embedding.shape))

## 10. 10 · 새 사진으로 유사도 판별


In [ ]:
test_uploaded = files.upload()  # 별도의 테스트 사진 1장 선택
test_name = next(iter(test_uploaded))
with Image.open(test_name) as image:
    test_face = detector(image.convert("RGB"))
if test_face is None:
    print("테스트 사진에서 얼굴을 찾지 못했습니다.")
else:
    with torch.no_grad():
        test_vector = facenet(test_face.unsqueeze(0).to(device)).squeeze(0).cpu()
        test_vector = F.normalize(test_vector, dim=0)
    score = torch.dot(owner_embedding, test_vector).item()
    threshold = 0.70  # 교육용 예시값: 검증사진으로 조정해야 함
    print(f"유사도: {score:.3f}, 결과: {'OWNER' if score >= threshold else 'UNKNOWN'}")

## 11. 11 · 주인 대표벡터 저장


In [ ]:
model_path = "owner_embedding.pt"
torch.save(owner_embedding, model_path)
print("저장 완료:", model_path)
# 얼굴 특징벡터도 생체정보이므로 공개 GitHub에 업로드하지 마세요.

## 12. 12 · Windows 웹캠 실습으로 연결


In [ ]:
files.download("owner_embedding.pt")
print("다운로드한 owner_embedding.pt를 로컬 프로젝트의 data/ 폴더에 넣으세요.")
print("Windows: pip install -r requirements.txt")
print("Windows: python src/webcam.py --embedding data/owner_embedding.pt")
print("웹캠 창에서 Q를 누르면 종료됩니다.")